<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Máquinas de vectores de soporte (SVM)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 📐 SVM: teoría, implementación desde cero y evaluación

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Clasificar manzanas y naranjas según peso y tamaño con una SVM. |
| 🔍 Comprensión de los datos | Cargamos `apples_and_oranges.csv` y repasamos la teoría del margen máximo. |
| 🧹 Preparación de los datos | Estandarizamos las variables, codificamos la clase como −1/+1 y dividimos 80/20. |
| 🤖 Modelado | Programamos una SVM lineal de margen blando y la comparamos con `SVC`; buscamos `C` por validación cruzada. |
| 📊 Evaluación | Exactitud, matriz de confusión y curva ROC. |
| 🚀 Despliegue y conclusiones | Reforzamos con kernels, `C` y `gamma`. |

> 🎯 **Analogía:** trazar una calle lo más ancha posible entre dos barrios; las casas que quedan al borde de la calle (los vectores de soporte) son las que definen por dónde pasa.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

### 📥 Datos del curso
Los archivos se descargan automáticamente (una sola vez) desde el repositorio del curso. Si ya los subió a `sample_data/`, no se vuelven a descargar.

In [ ]:
import os, urllib.request
REPO = 'https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main'
os.makedirs('sample_data', exist_ok=True)
for nombre, dia in [('apples_and_oranges.csv', 'dia_3')]:
    ruta = f'sample_data/{nombre}'
    if not os.path.exists(ruta):
        urllib.request.urlretrieve(f'{REPO}/{dia}/{nombre}', ruta)
        print('Descargado:', ruta)
    else:
        print('Ya existe:', ruta)

---
## 🎯 Fase 1 · Comprensión del negocio

Clasificar manzanas y naranjas según peso y tamaño con una SVM.

📚 Lectura recomendada: [3]

### 🧭 Algoritmos de clasificación del curso

Árbol de decisión · Bosque aleatorio · k vecinos más próximos · **Máquinas de vectores de soporte (SVM)**.

---
## 🔍 Fase 2 · Comprensión de los datos — teoría y datos

Cargamos `apples_and_oranges.csv` y repasamos la teoría del margen máximo.

📚 Lectura recomendada: [3] [4]

# **Máquinas de vectores de soporte (SVM)**

La SVM se desarrolló en los años 60 y se refinó en los 90 [3]. Es un algoritmo **supervisado** que sirve para clasificación, regresión y detección de valores atípicos, y suele rendir bien aun con pocos datos.

![SVM: ¿qué hiperplano elegir? Se maximiza el margen entre las clases](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_09.png)
*Figura: SVM: ¿qué hiperplano elegir? Se maximiza el margen entre las clases. [11]*

La SVM busca el **hiperplano** que separa las clases con el **mayor margen**. Los puntos más cercanos a la frontera son los **vectores de soporte**. La ecuación del hiperplano es

$$ w^{T}x + b = 0 $$

y la regla de decisión es $h(x) = +1$ si $w^{T}x + b \geq 0$ y $-1$ en otro caso. El **margen** es $2/\lVert w\rVert$, así que maximizar el margen equivale a minimizar $\tfrac12\lVert w\rVert^{2}$.

### Margen duro y margen blando

- **Margen duro (hard margin):** exige separar perfectamente las clases; solo existe si son linealmente separables y es muy sensible a valores atípicos.
- **Margen blando (soft margin):** permite algunos errores penalizados por el parámetro **C** [3].

$$ \min_{w,b}\ \tfrac12\lVert w\rVert^{2} + C\sum_i \max(0,\,1 - y_i(w^{T}x_i + b)) $$

![Efecto del parámetro C: margen estrecho (C grande) vs. margen amplio (C pequeño)](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_10.png)
*Figura: Efecto del parámetro C: margen estrecho (C grande) vs. margen amplio (C pequeño). [11]*

### Parámetro **C**

C pondera el equilibrio entre un margen amplio y pocos errores. **C pequeño:** margen ancho, tolera errores (más sesgo). **C grande:** margen estrecho, castiga errores (más varianza, riesgo de sobreajuste). Se elige por **validación cruzada**.

### SVM no lineal

Si las clases no se separan con una recta, un **kernel** proyecta los datos a un espacio de más dimensiones [4].

![Truco del kernel: datos no separables en 2D se vuelven separables en 3D](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_13.png)
*Figura: Truco del kernel: datos no separables en 2D se vuelven separables en 3D. [11]*

### Código desde cero: SVM lineal de margen blando

Se minimiza la función de costo anterior por **descenso de gradiente**. Requiere etiquetas **−1 / +1** (no 0 / 1), por eso se recodifican antes de entrenar.

---
## 🤖 Fase 4 · Modelado

Programamos una SVM lineal de margen blando y la comparamos con `SVC`; buscamos `C` por validación cruzada.

📚 Lectura recomendada: [3]

In [ ]:
class LinearSVMUsingSoftMargin:
    def __init__(self, C=1.0):
        self._support_vectors = None
        self.C = C
        self.beta = None
        self.b = None
        self.X = None
        self.y = None

        # n is the number of data points
        self.n = 0

        # d is the number of dimensions
        self.d = 0

    def __decision_function(self, X):
        return X.dot(self.beta) + self.b

    def __cost(self, margin):
        return (1 / 2) * self.beta.dot(self.beta) + self.C * np.sum(np.maximum(0, 1 - margin))

    def __margin(self, X, y):
        return y * self.__decision_function(X)

    def fit(self, X, y, lr=1e-3, epochs=500):
        # Initialize Beta and b
        self.n, self.d = X.shape
        self.beta = np.random.randn(self.d)
        self.b = 0

        # Required only for plotting
        self.X = X
        self.y = y

        loss_array = []
        for _ in range(epochs):
            margin = self.__margin(X, y)
            loss = self.__cost(margin)
            loss_array.append(loss)

            misclassified_pts_idx = np.where(margin < 1)[0]
            d_beta = self.beta - self.C * y[misclassified_pts_idx].dot(X[misclassified_pts_idx])
            self.beta = self.beta - lr * d_beta

            d_b = - self.C * np.sum(y[misclassified_pts_idx])
            self.b = self.b - lr * d_b

        self._support_vectors = np.where(self.__margin(X, y) <= 1)[0]

    def predict(self, X):
        return np.sign(self.__decision_function(X))

    def score(self, X, y):
        P = self.predict(X)
        return np.mean(y == P)

    def plot_decision_boundary(self):
        plt.scatter(self.X[:, 0], self.X[:, 1], c=self.y, s=50, cmap=plt.cm.Paired, alpha=.7)
        ax = plt.gca()
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()

        # create grid to evaluate model
        xx = np.linspace(xlim[0], xlim[1], 30)
        yy = np.linspace(ylim[0], ylim[1], 30)
        YY, XX = np.meshgrid(yy, xx)
        xy = np.vstack([XX.ravel(), YY.ravel()]).T
        Z = self.__decision_function(xy).reshape(XX.shape)

        # plot decision boundary and margins
        ax.contour(XX, YY, Z, colors=['r', 'b', 'r'], levels=[-1, 0, 1], alpha=0.5,
                   linestyles=['--', '-', '--'], linewidths=[2.0, 2.0, 2.0])

        # highlight the support vectors
        ax.scatter(self.X[:, 0][self._support_vectors], self.X[:, 1][self._support_vectors], s=100,
                   linewidth=1, facecolors='none', edgecolors='k')

        plt.show()

---
## 🧹 Fase 3 · Preparación de los datos

Estandarizamos las variables, codificamos la clase como −1/+1 y dividimos 80/20.

📚 Lectura recomendada: [5]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC

np.random.seed(0)
data = pd.read_csv('sample_data/apples_and_oranges.csv')
X = StandardScaler().fit_transform(data[['Weight', 'Size']])
le = LabelEncoder()
y = 2 * le.fit_transform(data['Class']) - 1          # recodificar 0/1 -> -1/+1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=1, stratify=y)
print('Clases:', list(le.classes_), '-> -1 / +1')

---
## 🤖 Fase 4 · Modelado

Programamos una SVM lineal de margen blando y la comparamos con `SVC`; buscamos `C` por validación cruzada.

In [ ]:
model = LinearSVMUsingSoftMargin(C=2)
model.fit(X_train, y_train)
print('Exactitud en prueba (SVM desde cero):', model.score(X_test, y_test))
print('Exactitud en prueba (SVC de sklearn): ', SVC(kernel='linear', C=2).fit(X_train, y_train).score(X_test, y_test))
model.plot_decision_boundary()

In [ ]:
# Definir el rango de valores de C a probar
param_grid = {'C': [0.1, 1, 10, 100]}

# Crear una instancia del modelo de SVM
svc = SVC(kernel='linear', gamma='auto')

# Crear una instancia de GridSearchCV para validar diferentes valores de C
grid_search = GridSearchCV(svc, param_grid, cv=5)

# Entrenar el modelo con GridSearchCV
grid_search.fit(X_train, y_train)

# Imprimir los resultados de la validación cruzada
print("Mejor valor de C encontrado: ", grid_search.best_params_)
print("Exactitud promedio en validación cruzada: ", grid_search.best_score_)


In [ ]:
C_values = grid_search.cv_results_['param_C'].data.astype(float)
accuracy_values = grid_search.cv_results_['mean_test_score']

# Graficamos los resultados
plt.semilogx(C_values, accuracy_values)
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('Validation Curve with SVM')
plt.show()

---
## 📊 Fase 5 · Evaluación

Exactitud, matriz de confusión y curva ROC.

📚 Lectura recomendada: [8] [9]

### **Curva ROC**

La curva ROC (*Receiver Operating Characteristic*) grafica la **sensibilidad** frente a **1 − especificidad** al variar el umbral de decisión [8]; el área bajo la curva (AUC) resume el desempeño.

![Curvas ROC con su AUC y la línea de adivinar al azar](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_10.png)
*Figura: Curvas ROC con su AUC y la línea de adivinar al azar. [11]*

### **Matriz de confusión**

![Matriz de confusión: VP, FN, FP y VN](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch06/images/06_08.png)
*Figura: Matriz de confusión: VP, FN, FP y VN. [11]*

Permite leer verdaderos/falsos positivos y negativos y derivar sensibilidad y especificidad [9].

---
## 🧪 Refuerzo: kernels, C y gamma: cuándo una frontera recta no basta

Una SVM lineal solo traza rectas (o hiperplanos). Si las clases se mezclan en forma curva (por ejemplo, dos lunas), hace falta un **kernel** que lleve los datos a un espacio donde sí sean separables [4]. Con el kernel **RBF** aparece un segundo hiperparámetro, **gamma**: gamma grande = frontera muy ajustada a cada punto (sobreajuste); gamma pequeño = frontera suave. Se comparan kernel lineal y RBF con distintos `C` y `gamma` sobre el conjunto sintético *moons*, usando validación cruzada de 5 particiones [5].

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=0)
modelos = {
    'lineal, C=1':          SVC(kernel='linear', C=1),
    'RBF, C=1, gamma=0.1':  SVC(kernel='rbf', C=1, gamma=0.1),
    'RBF, C=1, gamma=1':    SVC(kernel='rbf', C=1, gamma=1),
    'RBF, C=100, gamma=50': SVC(kernel='rbf', C=100, gamma=50),
}
for nombre, m in modelos.items():
    pipe = make_pipeline(StandardScaler(), m)
    pipe.fit(Xm, ym)
    cv = cross_val_score(pipe, Xm, ym, cv=5).mean()
    print(f'{nombre:22s} exactitud en entrenamiento = {pipe.score(Xm, ym):.3f} | validación cruzada = {cv:.3f}')

### ✍️ Ejercicios
1. ¿Qué modelo mejora más entre entrenamiento y validación cruzada? Eso indica **sobreajuste**.
2. Busque con `GridSearchCV` la mejor combinación de `C` y `gamma`.
3. Pruebe `kernel='poly'` con `degree=3`. ¿Cuándo preferiría un kernel polinomial sobre RBF? [7]

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score, classification_report

# Manzanas vs. naranjas: matriz de confusión y curva ROC del modelo de sklearn
svc_lin = SVC(kernel='linear', C=grid_search.best_params_['C']).fit(X_train, y_train)
pred = svc_lin.predict(X_test)
print(classification_report(y_test, pred, target_names=list(le.classes_)))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=le.classes_).plot()
plt.show()

puntaje = svc_lin.decision_function(X_test)           # la ROC se calcula con puntajes, no con etiquetas
fpr, tpr, _ = roc_curve(y_test, puntaje)
plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, puntaje):.3f}'); plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('Tasa de falsos positivos'); plt.ylabel('Tasa de verdaderos positivos'); plt.legend(); plt.show()

---
## ✅ Conclusiones y siguientes pasos

La SVM desde cero y `SVC` de scikit-learn coinciden (exactitud 1.0), pero el conjunto de prueba tiene solo **8 observaciones** y las clases son casi separables, así que ese 100 % no demuestra gran capacidad de generalización: conviene validación cruzada [5]. En *moons* se ve el papel del kernel y de los hiperparámetros: el modelo lineal llega a 0.84, el RBF con gamma = 1 a 0.96 y el RBF con C = 100 y gamma = 50 memoriza el entrenamiento (1.00) pero baja a 0.92 en validación, señal de sobreajuste [4].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [6], [10].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning, 20*(3), 273–297. https://doi.org/10.1007/BF00994018

**[4]** Boser, B. E., Guyon, I. M., & Vapnik, V. N. (1992). A training algorithm for optimal margin classifiers. *Proceedings of the 5th Annual Workshop on Computational Learning Theory (COLT '92)*, 144–152. https://doi.org/10.1145/130385.130401

**[5]** Hsu, C.-W., Chang, C.-C., & Lin, C.-J. (2003). *A practical guide to support vector classification*. Department of Computer Science, National Taiwan University. https://www.csie.ntu.edu.tw/~cjlin/papers/guide/guide.pdf

**[6]** Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning: Data Mining, Inference, and Prediction* (2.ª ed.). Springer. https://doi.org/10.1007/978-0-387-84858-7

**[7]** Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3.ª ed.). O'Reilly Media.

**[8]** Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters, 27*(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

**[9]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[10]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[11]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)